In [1]:
%%time

import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, make_scorer, mean_absolute_error
from sklearn.pipeline import make_pipeline,make_union
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score, KFold, ShuffleSplit, learning_curve, GridSearchCV
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')


Wall time: 1min 2s


In [2]:
train_df = pd.read_csv('data/Train.csv')
train_df.drop(["ID"], axis=1, inplace=True)

In [3]:
df = train_df.copy()

In [4]:
df.shape

(3870, 43)

In [5]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


RMSE=make_scorer(rmse)

In [6]:
def create_date_features(data, feature):
    return pd.DatetimeIndex(data[feature]).month.map(str)


def create_duration_features(data, features):
    duration_df = pd.DataFrame()
    data[features]=data[features].apply(pd.to_datetime)
    patterns=[i.split("/") for i in (set(["/".join(sorted(list(set([i,j])))) for i in features for j in features if len(set([i,j]))>1]))]
    for i in patterns:
        duration_df[f"{i[0]}-{i[1]}"]=(data[i[1]]-data[i[0]]).dt.days
    return duration_df

In [7]:
def create_new_features(df):
    features=["CropTillageDate","RcNursEstDate","SeedingSowingTransplanting","Harv_date","Threshing_date"]

    df["CropTillageMonth"]=create_date_features(df, "CropTillageDate")
    df["RcNursEstMonth"]=create_date_features(df, "RcNursEstDate")
    df["SeedingSowingTransplantingMonth"]=create_date_features(df,"SeedingSowingTransplanting")
    df["Harv_Month"]=create_date_features(df, "Harv_date")
    #df["Threshing_Month"]=create_date_features(df, "Threshing_date")

    duration_df=create_duration_features(df.copy(), features)

    df = pd.concat([df, duration_df], axis=1)

    df = df.drop(['Harv_date-RcNursEstDate'],axis=1)

    #df.dropna(subset=["CropOrgFYM"], axis=0)

    df = df[df.columns[df.isna().sum()/df.shape[0] < 0.2 ]]

    #df = df.select_dtypes(np.number)

    return df

In [8]:
process = 0

def imputation(df, i=1):

    global process
    ### HANDLING MISSING DATA ###

    # cat_features = df.select_dtypes(np.object_).columns.to_list()
    # num_features = df.select_dtypes(np.number).columns.to_list()
    # num_features.remove("Yield")

    # df[cat_features]=df[cat_features].fillna(df.mode().iloc[0])
    # df[num_features]=df[num_features].fillna(0)

    if i==1:
        numerical_features=df.select_dtypes(np.number).columns
        categorical_features=df.select_dtypes(np.object_).columns
        
        numerical_pipeline=make_pipeline(SimpleImputer(strategy="mean"))
        #numerical_pipeline=make_union(SimpleImputer(strategy="constant", fill_value=1234), MissingIndicator())
        categorical_pipeline=make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown='ignore'))
        process=make_column_transformer((numerical_pipeline, numerical_features), (categorical_pipeline, categorical_features))
        process.fit(df)
        data=process.transform(df)
    else:
        data=process.transform(df)
    
    return data

In [9]:
def encodage(df):
    return df

In [10]:
def pre_processing(df):

    df=create_new_features(df)
    #df=encodage(df)

    if "Yield" in df.columns.values:
        X = imputation(df.drop("Yield",axis=1))
        y = df.Yield
        return X,y
    else:
        X = imputation(df)
        return X

In [11]:
X,y=pre_processing(df)

In [12]:
X.shape

(3870, 918)

In [13]:
y.shape

(3870,)

In [12]:
def rmse_lgb(y_pred, y_true):
    
    rate=mean_squared_error(y_true, np.expm1(y_pred), squared=False)
    
    return "rmse_lgb", rate, False

In [13]:
RANDOM_SEED=0
NUM_THREADS=0
MODEL_PARAMS_LGB = {
    "boosting_type": 'gbdt',
    "objective": "regression",  # <= set objective
    "metric":None,
    "seed": RANDOM_SEED,
    "verbose": -1,
}   
FIT_PARAMS_LGB = {"early_stopping_rounds": 100, "verbose_eval":100}

cv=KFold(n_splits=5,shuffle=True,random_state=RANDOM_SEED)
scores=[]
for i,j in enumerate(cv.split(X,y)):
    model=LGBMRegressor(boosting_type='gbdt', objective='regression',seed=RANDOM_SEED)
    train_X=X[j[0]]
    train_y=np.log1p(y.iloc[j[0]])
    test_X=X[j[1]]
    test_y=np.log1p(y.iloc[j[1]])
    model.fit(train_X,train_y,
        eval_names=['train', 'valid'],
        eval_set=[(train_X,train_y),(test_X, test_y)], 
        eval_metric=rmse_lgb,
        verbose = False)
    
    y_pred=model.predict(X[j[1]])
    score=rmse_lgb(y_pred,y.iloc[j[1]])
    scores.append(score[1])
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:('rmse_lgb', 377.9280550529019, False)

Fold 1	:('rmse_lgb', 795.008938452372, False)

Fold 2	:('rmse_lgb', 464.90478928457406, False)

Fold 3	:('rmse_lgb', 315.12713311027545, False)

Fold 4	:('rmse_lgb', 173.94867769155275, False)

CV_SCORES_mean: 425.3835187183352
CV_SCORES_std: 207.75603607209612


In [20]:
for i in cv.split(X,y):
    print(i)
    break

(array([   0,    2,    3, ..., 3866, 3867, 3868]), array([   1,    4,    5,   10,   14,   15,   22,   30,   33,   34,   36,
         39,   45,   49,   57,   64,   66,   71,   85,   88,   92,  104,
        117,  118,  124,  134,  142,  148,  156,  159,  169,  170,  177,
        182,  185,  188,  189,  191,  192,  195,  196,  202,  211,  214,
        218,  219,  224,  226,  227,  229,  234,  248,  252,  253,  263,
        276,  290,  295,  296,  298,  299,  302,  304,  306,  308,  311,
        316,  326,  333,  346,  347,  351,  352,  361,  369,  378,  380,
        381,  383,  398,  399,  402,  408,  410,  418,  436,  443,  452,
        453,  457,  463,  465,  467,  472,  473,  480,  481,  482,  485,
        489,  491,  496,  499,  500,  501,  502,  514,  517,  519,  520,
        521,  526,  528,  530,  531,  533,  534,  536,  538,  541,  543,
        545,  546,  553,  558,  562,  565,  566,  567,  569,  572,  575,
        581,  582,  587,  596,  598,  599,  615,  616,  621,  634,  635,


In [64]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state = 1234)

model=LGBMRegressor(boosting_type='gbdt', objective='regression',metric='mae', use_missing=True)
# model.fit(X_train,y_train,
#     eval_names=['train', 'valid'],
#     eval_set=[(X_train,y_train),(X_test, y_test)], 
#     eval_metric=rmse_lgb,
#     verbose = False)

model.fit(X,y,verbose = False)

LGBMRegressor(metric='mae', objective='regression', use_missing=True)

In [65]:
params = {
    "num_leaves": 1200,
    "learning_rate":0.17, 
    "n_estimators":30, 
    "max_depth":16, 
    "bagging_fraction":0.8, 
    "feature_fraction":0.8, 
    "reg_lambda":0.9, 
    "use_missing":False
}

## Submission

In [66]:
sub = pd.read_csv("data/SampleSubmission.csv")
test = pd.read_csv("data/Test.csv")
test.drop(["ID"], axis=1, inplace=True)

In [67]:
test=create_new_features(test)

In [68]:
test.shape

(1290, 47)

In [69]:
process.transform(test)

<1290x761 sparse matrix of type '<class 'numpy.float64'>'
	with 60551 stored elements in Compressed Sparse Row format>

In [70]:
test = pre_processing(test)
predict = model.predict(test)

ValueError: Selected columns, ['CultLand', 'CropCultLand', 'CropTillageDepth', 'SeedlingsPerPit', 'TransplantingIrrigationHours', 'StandingWater', 'NoFertilizerAppln', 'BasalDAP', '1tdUrea', '1appDaysUrea', 'Harv_hand_rent', 'Residue_length', 'Residue_perc', 'Acre', 'SeedingSowingTransplanting-Threshing_date', 'SeedingSowingTransplanting-Threshing_date', 'CropTillageDate-Harv_date', 'CropTillageDate-Harv_date', 'CropTillageDate-RcNursEstDate', 'CropTillageDate-RcNursEstDate', 'CropTillageDate-Threshing_date', 'CropTillageDate-Threshing_date', 'Harv_date-Threshing_date', 'Harv_date-Threshing_date', 'CropTillageDate-SeedingSowingTransplanting', 'CropTillageDate-SeedingSowingTransplanting', 'Harv_date-SeedingSowingTransplanting', 'Harv_date-SeedingSowingTransplanting', 'RcNursEstDate-Threshing_date', 'RcNursEstDate-Threshing_date', 'RcNursEstDate-SeedingSowingTransplanting', 'RcNursEstDate-SeedingSowingTransplanting', 'SeedingSowingTransplanting-Threshing_date', 'SeedingSowingTransplantin